# first project

**signal b**\
nested cross validation: train con cross validation su 18, poi 4 per test interno\
**signal a**\
test esterno su dati che il modello non ha mai visto

### machine learning supervisionato

- ogni segnale è un file .mat, in cui sono contenute due matrici 22x16288: g_0 e g_1
- Per la fase di training del modello usiamo il segnale B
- g_0 e g_1 rappresentano le due classi: negativi e positivi. 

- **ogni riga (2 x 22) è un pattern (campione/osservazione), ogni colonna (16288) è una _feature_ (caratteristica).**

### recupero la matrice da file .mat

In [1]:
import numpy as np

g_0_b = np.loadtxt('data/g_0_signal_b.txt')
g_1_b = np.loadtxt('data/g_1_signal_b.txt')

### controllo shape

In [2]:
print("shape g_0")
print(g_0_b.shape)
print("shape g_1")
print(g_1_b.shape)

shape g_0
(22, 16288)
shape g_1
(22, 16288)


In [3]:
g_0_b[20].shape[0]

16288

### controllo spettro

In [4]:
import matplotlib.pyplot as plt

"""
Se vuoi stampare tutto
x = range(signal_a[20].shape[0])
y = signal_a[20]
"""

for i in range(32):
    plt.figure(figsize=(5,4), dpi=100)
    plt.ioff() 
    da = i*509
    a = i*509+509
    y0 = g_0_b[20][da:a]
    y1 = g_1_b[20][da:a]
    x = range(g_0_b[20][da:a].shape[0])
    
    plt.plot(x,y0)
    plt.plot(x,y1)
    #plt.savefig(f'outputs/g_0_E_g_1_signal_b/g_0_E_g_1_signal_b_{i}.png')
    plt.close()


### preparazione ingredienti

Dobbiamo addestrare un modello svm (Support Vector Machine). Dobbiamo fornirgli:
- i dati (le 16288 colonne)
- le risposte corrette (labels)

per questo concateniamo g_0 e g_1, ottenendo una matrice 44x16288; quindi formiamo un vettore delle risposte da 44 entrate: 22 zero e 22 uno. ciascuna riga del matricione è in corrispondenza con una riga del vettore labels.

In [5]:
# sono entrambe 22, ma così è più generale
N0 = g_0_b.shape[0]
N1 = g_1_b.shape[0]

In [6]:
# concateno g_0 e g_1
matricione = np.vstack((g_0_b, g_1_b))

In [7]:
# vettore delle risposte
labels = np.concatenate((np.zeros(N0), np.ones(N1)))
original_indices = np.arange(1, 45, step=1)

In [8]:
N_neg = N0
n_pos = N1

### `cross_val_score`

In [9]:
# modulo delle support vector machines 
from sklearn import svm
from sklearn.model_selection import cross_val_score

cv = 5 # default: 5 

- `cross_val_score` valuta lo score di un **estimator** usando il suo metodo interno `.score()`
- `svm.SVC` è l'**estimator** scelto, il suo _score_ è l'**accuracy**: la frazione di pattern correttamente classificati
$$\texttt{accuracy}(y, \hat{y}) = \frac{1}{n_\text{samples}} \sum_{i=0}^{n_\text{samples}-1} 1(\hat{y}_i = y_i)$$
- l'**error** è $1-\texttt{accuracy}(y, \hat{y})$

vogliamo usare la strategia di _cross_validation_ `KFold`:

<p align="center">
  <img src="images/KFold-scheme.png" alt="Schema k-fold cross-validation">
</p>

In [10]:
from sklearn.model_selection import KFold
cv_strategy = KFold(n_splits=6, shuffle=True, random_state=42) 

se viene passato un integer a `cv`, viene usata automaticamente la strategia `KFold`; noi la passiamo esplicitamente.

In [11]:
# bisogna dirgli di essere lineare, oppure usare LinearSVC
estimator = svm.SVC(kernel='linear', random_state=None)
scores = cross_val_score(estimator, matricione, labels, cv=cv_strategy)

`cross_val_score` chiama il metodo `.fit()` di `svm.SVC` sui dati di training, poi il suo metodo `.score()` sui dati di test. 

In [12]:
scores

array([0.75      , 0.625     , 0.57142857, 0.57142857, 0.42857143,
       0.57142857])

In [13]:
media = np.mean(scores)
dev_st = np.std(scores)

print(f"accuracy = {media:.4f} ± {dev_st:.4f}")

accuracy = 0.5863 ± 0.0947


### `cross_val_predict`

In [14]:
from sklearn.model_selection import cross_val_predict
predictions = cross_val_predict(estimator, matricione, labels, cv=cv_strategy)

In [15]:
predictions

array([1., 1., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1.,
       0., 0., 0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 1., 1., 0., 0., 1.,
       1., 0., 1., 0., 0., 0., 1., 1., 1., 0.])

In [16]:
positivi = predictions[22:]
negativi = predictions[:22]

In [17]:
num_p, num_n = 0, 0
for p, n in zip (positivi, negativi):
    if p == 1:
        num_p += 1
    if n == 0:
        num_n += 1
        
acc = (num_p + num_n)/44
print(acc)

0.5909090909090909


### Manuale

`KFold.split()` non restituisce gli `n_split` array pronti all'uso: **genera** gli indici per un fold ogni volta che viene chiamato, per un totale di `n_split` volte.

> 💡 Provare `Stratified_KFold` per assicurarsi un buon bilanciamento di 0 e 1 in ogni fold. Perché *shuffle* di `KFold` non lo garantisce.

> 💡 Provare a implementare **Principal Component Analysis** (PCA)

In [ ]:
from sklearn import svm
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import accuracy_score # forse ridondante

# scelta dei seed
seed_list = [1_434_512, 432_632, 542_397, 76_301, 97, 7_233, 19_000_234, 500_634, 1, 3242]
kfold_seed = 42

fold_accuracies = {}
mean_accuracies = {}
std_accuracies = {}

for seed in seed_list:

    # istanza modello ⚠️ random_state qui non fa niente
    my_svm = svm.SVC(kernel='linear', random_state=42)
    # scelta della strategia di cross-validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed) 

    # lista per accuracies
    fold_accuracies[seed] = []
 
    for i, (train_idx, test_idx) in enumerate( kf.split(matricione, labels) ): 
        
        pattern_train, pattern_test = matricione[train_idx], matricione[test_idx]
        labels_train, labels_test = labels[train_idx], labels[test_idx]
        
        # fit
        my_svm.fit(pattern_train, labels_train)
        
        # predict
        pred = my_svm.predict(pattern_test)
        
        # score (accuracy)
        acc = accuracy_score(labels_test, pred)
        #sco = my_svm.score(pattern_test, labels_test) # metodo .score di SVC dà l'accuracy di default!
        fold_accuracies[seed].append(acc)
        
        #print(f"Fold {i+1}: Accuracy = {acc:.5f} | SVC.score = {sco:.5f}")
    mean_accuracies[seed] = np.mean(fold_accuracies[seed])
    std_accuracies[seed] = np.std(fold_accuracies[seed])

    for s in fold_accuracies[seed]: print(f"{s:.3f}", end=', ')
    print(f"Accuracy media sui fold: {mean_accuracies[seed]:.5f} ± {std_accuracies[seed]:.5f}")
    
    
total_mean = np.mean(list(mean_accuracies.values()))
total_std = np.std(list(mean_accuracies.values()))
print(f"Accuracy media sui seed delle medie sui fold: {total_mean:.3f} ± {total_std:.3f}")

0.444, 0.667, 0.556, 0.444, 0.625, Accuracy media sui fold: 0.54722 ± 0.09112
0.667, 0.667, 0.778, 0.667, 0.750, Accuracy media sui fold: 0.70556 ± 0.04843
0.778, 0.889, 0.667, 0.444, 0.625, Accuracy media sui fold: 0.68056 ± 0.14959
0.667, 0.667, 0.556, 0.444, 0.500, Accuracy media sui fold: 0.56667 ± 0.08889
0.667, 0.556, 0.667, 0.556, 0.625, Accuracy media sui fold: 0.61389 ± 0.05000
0.778, 0.556, 0.667, 0.667, 0.875, Accuracy media sui fold: 0.70833 ± 0.10901
0.556, 0.667, 0.889, 0.667, 0.500, Accuracy media sui fold: 0.65556 ± 0.13333
1.000, 0.667, 0.556, 0.667, 0.750, Accuracy media sui fold: 0.72778 ± 0.14948
0.444, 0.444, 0.778, 0.556, 0.625, Accuracy media sui fold: 0.56944 ± 0.12485
0.333, 0.333, 0.778, 0.556, 0.750, Accuracy media sui fold: 0.55000 ± 0.19277
Accuracy media sui seed delle medie sui fold: 0.6325000000000001 ± 0.06763247174669795


Cos'è il segnale? Si può lavorare in un altro spazio? Benchmark

### GridSearch

In [ ]:
# parametri da provare
param_grid = [
  {'C': [1E-07, 1E-06, 1E-05, 1E-04], 'kernel': ['linear']},
]

In [42]:
from sklearn import svm
from sklearn.model_selection import StratifiedKFold, GridSearchCV

from sklearn.metrics import accuracy_score # forse ridondante

# scelta dei seed
seed_list = [1_434_512, 432_632, 542_397, 76_301, 97, 7_233, 19_000_234, 500_634, 1, 3242]
kfold_seed = 42

best_c = {}
best_accuracies = {}

for seed in seed_list:
    print(f"🚀 SEED: {seed}")
    # istanza modello ⚠️ random_state qui non fa niente
    my_svm = svm.SVC()#kernel='linear', random_state=42)
    # scelta della strategia di cross-validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed) 
    
    # gridsearch: proverà diverse combinazioni di parametri
    gs = GridSearchCV(my_svm, cv=kf, param_grid=param_grid)
    
    # fit
    gs.fit(matricione, labels)
    
    # controllo parametri
    print(" - Migliori iperparametri:", gs.best_params_)
    print(" - Miglior score:", gs.best_score_)

    best_c[seed] = gs.best_params_['C']
    best_accuracies[seed] = gs.best_score_
    
media_accuracies = np.mean(list(best_accuracies.values()))
std_accuracies = np.std(list(best_accuracies.values()))
print(f"MEDIA: {media_accuracies} ± {std_accuracies}")

🚀 SEED: 1434512
 - Migliori iperparametri: {'C': 1e-07, 'kernel': 'linear'}
 - Miglior score: 0.6416666666666667
🚀 SEED: 432632
 - Migliori iperparametri: {'C': 1e-05, 'kernel': 'linear'}
 - Miglior score: 0.7055555555555555
🚀 SEED: 542397
 - Migliori iperparametri: {'C': 1e-05, 'kernel': 'linear'}
 - Miglior score: 0.6805555555555556
🚀 SEED: 76301
 - Migliori iperparametri: {'C': 1e-07, 'kernel': 'linear'}
 - Miglior score: 0.5916666666666667
🚀 SEED: 97
 - Migliori iperparametri: {'C': 1e-06, 'kernel': 'linear'}
 - Miglior score: 0.6138888888888889
🚀 SEED: 7233
 - Migliori iperparametri: {'C': 1e-05, 'kernel': 'linear'}
 - Miglior score: 0.7083333333333333
🚀 SEED: 19000234
 - Migliori iperparametri: {'C': 1e-05, 'kernel': 'linear'}
 - Miglior score: 0.6555555555555556
🚀 SEED: 500634
 - Migliori iperparametri: {'C': 1e-05, 'kernel': 'linear'}
 - Miglior score: 0.7277777777777777
🚀 SEED: 1
 - Migliori iperparametri: {'C': 1e-07, 'kernel': 'linear'}
 - Miglior score: 0.638888888888889
🚀 